# 03 — Train LightGBM Baseline / เทรน Baseline LightGBM

**EN.** Train LightGBM regression baselines for every `(station_id, horizon_h)` pair using `configs/train/baseline.yaml`. Runs are **resumable** — each successful pair writes its v3 bundle plus a manifest entry, so a Colab disconnect at hour 11 does not lose hour 1–10 of work. Re-running this notebook skips pairs that are already complete.

**TH.** เทรน baseline LightGBM ทุก `(station_id, horizon_h)` ตามคอนฟิก `configs/train/baseline.yaml`. รองรับการ **resume** — ทุกคู่ที่เทรนสำเร็จจะเขียน v3 bundle + manifest entry ทันที ดังนั้นถ้า Colab หลุดที่ชั่วโมงที่ 11 จะไม่เสีย 10 ชั่วโมงแรก. รันซ้ำ notebook นี้จะข้ามคู่ที่เสร็จแล้ว.

**Outputs / ผลลัพธ์**

- v3 artifacts: `app/models/forecast_v3/{station_id}/h{H}/{bundle.json,registry.json,...}`
- Choice matrix: `app/models/forecast_v3/choice_matrix.json` (entry `lightgbm_hi_quantile` per pair)
- Run snapshot: `runs/{RUN_ID}/config.yaml`, `runs/{RUN_ID}/manifest.json`, `runs/{RUN_ID}/metrics.csv`
- Inline mini-report: per-pair MAE/RMSE/Bias/Peak-hour MAE table + per-station MAE bar chart

## 1. Setup / ตั้งค่า

**EN.** Re-run the Colab bootstrap so the repo is cloned, Drive is mounted, and `data/` + `app/models/forecast_v3/` symlinks point at Drive. Idempotent — safe to re-run.

**TH.** รัน bootstrap เพื่อ clone repo, mount Drive และทำ symlink `data/` + `app/models/forecast_v3/` ไปยัง Drive. รันซ้ำได้โดยปลอดภัย.

In [ ]:
# --- bootstrap (idempotent) ---------------------------------------------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if os.path.exists(f"{REPO_DIR}/scripts/colab_bootstrap.sh"):
    !bash {REPO_DIR}/scripts/colab_bootstrap.sh
else:
    print("[setup] bootstrap script not found — assuming local Jupyter run.")
    REPO_DIR = os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")

## 2. Config + manifest / โหลดคอนฟิกและ manifest

**EN.** Load `configs/train/baseline.yaml`, derive `RUN_ID = {YYYYMMDD}_baseline_lgbm`, snapshot the config under `runs/{RUN_ID}/config.yaml`, and prepare/refresh `manifest.json`. A pair is treated as **complete** when both the manifest says so AND `app/models/forecast_v3/{station}/h{H}/bundle.json` exists.

**TH.** โหลด `configs/train/baseline.yaml`, ตั้ง `RUN_ID = {YYYYMMDD}_baseline_lgbm`, snapshot คอนฟิกไว้ที่ `runs/{RUN_ID}/config.yaml`, และเตรียม/อัปเดต `manifest.json`. คู่ `(station, horizon)` จะถือว่า **เสร็จ** เมื่อ manifest บอกว่าเสร็จ **และ** มีไฟล์ `bundle.json` แล้ว.

In [ ]:
# --- load YAML config + build run dir + manifest -------------------------
import json, shutil, datetime as _dt
from pathlib import Path
import yaml

CONFIG_PATH = Path("configs/train/baseline.yaml")
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

STATIONS_CFG   = list(CFG["stations"])
HORIZONS_CFG   = list(CFG["horizons"])
DATA_WINDOW    = CFG["data_window"]              # [start, end] ISO strings or dates
LGBM_PARAMS    = dict(CFG["params"])             # base hyperparams
TRAINING_CFG   = dict(CFG["training"])           # num_boost_round, early_stopping_rounds, cv_folds, cv_gap_h
OPTUNA_CFG     = dict(CFG.get("optuna", {}))
ARTIFACTS_CFG  = dict(CFG.get("artifacts", {}))
SEED           = int(CFG.get("seed", 42))
TARGET_KIND    = "hi" if str(CFG.get("target", "hi_t_plus_h")).startswith("hi") else "th"

_today = _dt.date.today().strftime("%Y%m%d")
RUN_ID = f"{_today}_baseline_lgbm"
RUN_DIR = Path("runs") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Snapshot the YAML (verbatim copy — no transformation, so we can diff later).
shutil.copy2(CONFIG_PATH, RUN_DIR / "config.yaml")

# Build run list (closed product of stations x horizons).
RUNS = [(sid, int(h)) for sid in STATIONS_CFG for h in HORIZONS_CFG]

# Manifest: pair status across re-runs.
MANIFEST_PATH = RUN_DIR / "manifest.json"
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
else:
    manifest = {
        "run_id": RUN_ID,
        "backend": CFG["backend"],
        "target": CFG.get("target", "hi_t_plus_h"),
        "created_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "pairs": {},   # "{station}_h{H}" -> {"status": ..., "metrics": {...}, "saved_at_utc": ...}
    }

ARTIFACT_ROOT = Path(ARTIFACTS_CFG.get("out_root", "app/models/forecast_v3"))

def _bundle_path(station_id: str, horizon_h: int) -> Path:
    return ARTIFACT_ROOT / station_id / f"h{horizon_h}" / "bundle.json"

def _is_complete(station_id: str, horizon_h: int) -> bool:
    key = f"{station_id}_h{horizon_h}"
    entry = manifest["pairs"].get(key, {})
    return entry.get("status") == "complete" and _bundle_path(station_id, horizon_h).exists()

PENDING = [(s, h) for (s, h) in RUNS if not _is_complete(s, h)]
DONE    = [(s, h) for (s, h) in RUNS if _is_complete(s, h)]

print(f"RUN_ID    : {RUN_ID}")
print(f"backend   : {CFG['backend']}  (target_kind resolved to '{TARGET_KIND}')")
print(f"stations  : {STATIONS_CFG}")
print(f"horizons  : {HORIZONS_CFG}")
print(f"window    : {DATA_WINDOW[0]} -> {DATA_WINDOW[1]}")
print(f"runs total: {len(RUNS)}  | done: {len(DONE)}  | pending: {len(PENDING)}")

# Persist manifest scaffold so the file always exists from cell 2 onward.
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

## 3. Data load / โหลดข้อมูล

**EN.** Read parquet observations through `app.data.loaders.read_observations` for the configured `data_window`. The loader prefers TMD > ERA5 > NASA POWER for the same `(station_id, ts_utc)`. Cache once per station so we do not re-read parquet for every horizon.

**TH.** อ่าน parquet ผ่าน `app.data.loaders.read_observations` ตามช่วงใน `data_window`. Loader เลือก TMD > ERA5 > NASA POWER สำหรับ `(station_id, ts_utc)` เดียวกัน. โหลดครั้งเดียวต่อสถานีแล้วใช้ซ้ำกับทุก horizon.

In [ ]:
# --- read parquet observations per station ------------------------------
import datetime as _dt
import pandas as pd
from app.data.loaders import read_observations

def _to_date(value):
    if isinstance(value, _dt.date):
        return value
    return _dt.date.fromisoformat(str(value))

WINDOW_START = _to_date(DATA_WINDOW[0])
WINDOW_END   = _to_date(DATA_WINDOW[1])

OBS_BY_STATION: dict[str, pd.DataFrame] = {}
for sid in STATIONS_CFG:
    df = read_observations(sid, WINDOW_START, WINDOW_END)
    OBS_BY_STATION[sid] = df
    n = len(df)
    src = df["source"].value_counts().to_dict() if (n and "source" in df.columns) else {}
    print(f"  {sid}: {n:>7d} rows  | sources={src}")

_total_rows = sum(len(v) for v in OBS_BY_STATION.values())
if _total_rows == 0:
    raise RuntimeError(
        "No observations loaded — run 01_ingest.ipynb first or check data_window in baseline.yaml."
    )

## 4. Feature build / สร้างฟีเจอร์

**EN.** Build `X` once per station via `build_X_once`, then materialize `y` per horizon via `build_y_for_horizon`. The training-time feature builder enforces the no-leakage invariant (every row in `X` only uses observations with `ts_utc <= row.ts_utc`).

**TH.** สร้าง `X` ครั้งเดียวต่อสถานีด้วย `build_X_once` แล้วสร้าง `y` ต่อ horizon ด้วย `build_y_for_horizon`. ตัวสร้างฟีเจอร์รับประกัน no-leakage (`X[i]` ใช้เฉพาะข้อมูลที่ `ts_utc <= row.ts_utc`).

In [ ]:
# --- build features per station (cached) --------------------------------
import pandas as pd
from app.ml.forecast.features import build_X_once, build_y_for_horizon

FEATURES_BY_STATION: dict[str, dict] = {}

for sid, df in OBS_BY_STATION.items():
    if df.empty:
        print(f"  {sid}: empty observations — will skip all horizons.")
        continue
    work = df.copy()
    if "station_id" not in work.columns:
        work["station_id"] = sid
    X_full, df_aug = build_X_once(work)
    # Core (non-extended) features must be NaN-free; assert as a guard.
    core_cols = [c for c in X_full.columns if not any(c.startswith(p) for p in (
        "solar_wm2", "cloud_cover", "blh_m", "pressure_hpa", "lst_c",
        "wind_ms", "precip_mm",
    ))]
    assert not X_full[core_cols].isna().any().any(), f"{sid}: NaN found in core features"
    FEATURES_BY_STATION[sid] = {"X": X_full, "df_aug": df_aug}
    print(f"  {sid}: X shape = {X_full.shape}")

# Per-horizon shape preview (target rows after NaN-drop).
for sid, payload in FEATURES_BY_STATION.items():
    X_full = payload["X"]
    df_aug = payload["df_aug"]
    for h in HORIZONS_CFG:
        y_full = build_y_for_horizon(df_aug, X_full.index, int(h), target_kind=TARGET_KIND)
        if isinstance(y_full, pd.DataFrame):
            valid = y_full.notna().all(axis=1)
        else:
            valid = y_full.notna()
        print(f"  {sid} h{h}: rows={int(valid.sum()):>6d}  (target_kind={TARGET_KIND})")

## 5. Train loop with checkpoint resume / ลูปเทรนพร้อม checkpoint

**EN.** For each pending pair, fit a LightGBM forecaster from the v3 backend layer (`LGBMDirectHIForecaster` when `target == hi_t_plus_h`, otherwise `LGBMForecaster`). The backend internally handles chronological train/val/test split (via `app.ml.forecast.splitting.split_xy`, gap=`horizon_h`), Optuna search bounded by YAML `optuna.trials`, multi-seed quantile boosters, and conformal calibration. We then compute `(MAE, RMSE, Bias, Peak-hour MAE)` on the held-out test split, save artifacts via `registry.save_model_v3`, and update the manifest **immediately** so a Colab disconnect after this pair preserves the work.

**TH.** สำหรับแต่ละคู่ที่ยังไม่เสร็จ จะ fit LightGBM forecaster จาก v3 backend (`LGBMDirectHIForecaster` ถ้า `target == hi_t_plus_h`, ไม่งั้น `LGBMForecaster`). Backend จัดการ split เวลาให้ (gap = `horizon_h`), Optuna search ตาม YAML, quantile boosters หลาย seed, และ conformal calibration. หลัง fit คำนวณ `(MAE, RMSE, Bias, Peak-hour MAE)` บน test split, บันทึกผ่าน `registry.save_model_v3`, แล้ว **อัปเดต manifest ทันที** เพื่อให้รอด Colab disconnect.

In [ ]:
# --- per-pair training with immediate checkpointing ---------------------
import json, time
import numpy as np
import pandas as pd

from app.ml.forecast.features import build_y_for_horizon
from app.ml.forecast.splitting import split_xy
from app.ml.forecast.backends.lgbm_backend import (
    LGBMForecaster,
    LGBMDirectHIForecaster,
    _compute_hi_array,
)
from app.ml import registry

# Resolve forecaster class from the YAML target. Both classes consume the same
# baseline params + Optuna trial budget; the backend differs only in target shape.
_FORECASTER_CLS = LGBMDirectHIForecaster if TARGET_KIND == "hi" else LGBMForecaster
_OPTUNA_TRIALS = int(OPTUNA_CFG.get("trials", 30)) if OPTUNA_CFG.get("enabled", True) else 0
_MIN_ROWS = 500   # minimum usable training rows after target NaN-drop

# Local-hour bucket for peak-hour MAE: Thai daytime peak = 12:00–17:00 local.
_PEAK_LOCAL_HOURS = set(range(12, 18))

def _local_hour_from_X(X: pd.DataFrame) -> np.ndarray:
    """Reconstruct local hour (0..23) from the cyclic features the trainer used."""
    angles = np.arctan2(X["local_hour_sin"].values, X["local_hour_cos"].values)
    return np.round(angles * 24 / (2 * np.pi)).astype(int) % 24

def _save_manifest():
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

results: list[dict] = []
metrics_csv = RUN_DIR / "metrics.csv"
_csv_header = "station_id,horizon_h,n_train,n_val,n_test,mae,rmse,bias,peak_mae,train_seconds\n"
if not metrics_csv.exists():
    metrics_csv.write_text(_csv_header, encoding="utf-8")

for (station_id, horizon_h) in RUNS:
    pair_key = f"{station_id}_h{horizon_h}"

    # ---- skip if already complete on disk + manifest ----------------------
    if _is_complete(station_id, horizon_h):
        prior = manifest["pairs"].get(pair_key, {}).get("metrics", {})
        if prior:
            results.append({"station_id": station_id, "horizon_h": horizon_h, **prior})
        print(f"[skip] {pair_key} already complete")
        continue

    if station_id not in FEATURES_BY_STATION:
        manifest["pairs"][pair_key] = {"status": "skipped", "reason": "no_observations"}
        _save_manifest()
        print(f"[skip] {pair_key} — no observations")
        continue

    payload = FEATURES_BY_STATION[station_id]
    X_full = payload["X"]
    df_aug = payload["df_aug"]
    y_full = build_y_for_horizon(df_aug, X_full.index, horizon_h, target_kind=TARGET_KIND)
    if isinstance(y_full, pd.DataFrame):
        valid = y_full.notna().all(axis=1)
    else:
        valid = y_full.notna()
    X = X_full[valid]
    y = y_full[valid]

    if len(X) < _MIN_ROWS:
        manifest["pairs"][pair_key] = {
            "status": "skipped",
            "reason": f"too_few_rows ({len(X)} < {_MIN_ROWS})",
        }
        _save_manifest()
        print(f"[skip] {pair_key} — only {len(X)} rows")
        continue

    # ---- mark in-progress -------------------------------------------------
    started = time.perf_counter()
    manifest["pairs"][pair_key] = {
        "status": "in_progress",
        "started_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
    }
    _save_manifest()
    print(f"[fit ] {pair_key}  rows={len(X)}  trials={_OPTUNA_TRIALS}")

    # ---- fit (Optuna + multi-seed boosters all happen inside .fit) -------
    forecaster = _FORECASTER_CLS(n_trials=_OPTUNA_TRIALS, random_state=SEED)
    forecaster.fit(X, y, station_id=station_id, horizon_h=horizon_h)

    # ---- compute (MAE, RMSE, Bias, Peak-hour MAE) on the test split ------
    split = split_xy(X, y, horizon_h=horizon_h)
    bundle = forecaster.predict_with_pi(split.X_test)
    if TARGET_KIND == "th":
        y_true = _compute_hi_array(
            split.y_test["temp_c"].values, split.y_test["rh"].values
        )
    else:
        y_true = split.y_test.values
    y_pred = bundle.hi_mean
    err = y_pred - y_true
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    bias = float(np.mean(err))

    local_hours = _local_hour_from_X(split.X_test)
    peak_mask = np.isin(local_hours, list(_PEAK_LOCAL_HOURS))
    peak_mae = float(np.mean(np.abs(err[peak_mask]))) if peak_mask.any() else float("nan")

    train_seconds = float(time.perf_counter() - started)
    n_train = split.metadata["row_counts"]["train"]
    n_val   = split.metadata["row_counts"]["val"]
    n_test  = split.metadata["row_counts"]["test"]

    pair_metrics = {
        "mae": round(mae, 4),
        "rmse": round(rmse, 4),
        "bias": round(bias, 4),
        "peak_mae": round(peak_mae, 4) if peak_mae == peak_mae else None,
        "n_train": n_train,
        "n_val":   n_val,
        "n_test":  n_test,
        "train_seconds": round(train_seconds, 2),
    }

    # ---- save v3 artifacts (bundle.json + registry.json sidecar) ---------
    # registry.save_model_v3 writes forecaster.backend_name into both the
    # registry.json sidecar AND choice_matrix.json — do not override it.
    save_metadata = {
        "run_id": RUN_ID,
        "backend_yaml": CFG["backend"],
        "target_yaml": CFG.get("target", "hi_t_plus_h"),
        "baseline_params": LGBM_PARAMS,
        "training_cfg": TRAINING_CFG,
        "optuna_cfg": OPTUNA_CFG,
        "data_window": [str(WINDOW_START), str(WINDOW_END)],
        "metrics": pair_metrics,
        "split_metadata": split.metadata,
    }
    slot_dir = registry.save_model_v3(forecaster, save_metadata, station_id, horizon_h)

    # ---- checkpoint manifest ----------------------------------------------
    manifest["pairs"][pair_key] = {
        "status": "complete",
        "slot_dir": str(slot_dir),
        "backend_name": forecaster.backend_name,
        "saved_at_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
        "metrics": pair_metrics,
    }
    _save_manifest()

    with metrics_csv.open("a", encoding="utf-8") as fh:
        fh.write(
            f"{station_id},{horizon_h},{n_train},{n_val},{n_test},"
            f"{mae:.4f},{rmse:.4f},{bias:.4f},"
            f"{(peak_mae if peak_mae == peak_mae else float('nan')):.4f},"
            f"{train_seconds:.2f}\n"
        )

    results.append({"station_id": station_id, "horizon_h": horizon_h, **pair_metrics})
    print(
        f"[done] {pair_key}  MAE={mae:.3f}  RMSE={rmse:.3f}  "
        f"Bias={bias:+.3f}  PeakMAE={peak_mae:.3f}  ({train_seconds:.0f}s)"
    )

print("\nTraining loop complete. Manifest:", MANIFEST_PATH)

## 6. Verify saved artifacts / ตรวจสอบ artifact ที่บันทึกแล้ว

**EN.** Confirm every pair has a `bundle.json` (backend's metadata) **and** a `registry.json` sidecar (registry-level metadata) under `app/models/forecast_v3/{station}/h{H}/`. Backends own `bundle.json`; the registry writes the sidecar separately so saves never clobber backend fields.

**TH.** ยืนยันว่าทุกคู่มีทั้ง `bundle.json` (metadata จาก backend) และ `registry.json` (sidecar จาก registry) ใต้ `app/models/forecast_v3/{station}/h{H}/`. Backend เป็นเจ้าของ `bundle.json`; registry เขียน sidecar แยกเพื่อไม่ให้ทับฟิลด์ของ backend.

In [ ]:
# --- artifact verification ----------------------------------------------
from pathlib import Path

_missing = []
for (sid, h) in RUNS:
    slot = ARTIFACT_ROOT / sid / f"h{h}"
    bundle_ok   = (slot / "bundle.json").exists()
    registry_ok = (slot / "registry.json").exists()
    flag = "OK" if (bundle_ok and registry_ok) else "--"
    print(f"  [{flag}] {sid}/h{h}  bundle.json={bundle_ok}  registry.json={registry_ok}")
    if not (bundle_ok and registry_ok):
        _missing.append((sid, h))

matrix_path = ARTIFACT_ROOT / "choice_matrix.json"
if matrix_path.exists():
    import json as _json
    matrix = _json.loads(matrix_path.read_text())
    print("\nchoice_matrix.json:")
    print(_json.dumps(matrix, indent=2))
else:
    print("\nchoice_matrix.json missing — re-run train loop above.")

if _missing:
    print(f"\n{len(_missing)} pair(s) missing artifacts: {_missing}")

## 7. Inline mini-report / รายงานสรุปในเล่ม

**EN.** Per-pair metrics table (MAE / RMSE / Bias / Peak-hour MAE) + per-station MAE bar chart, rendered inline. The user requirement is explicit: **every train action must produce inline metrics + charts in the notebook**, not just on disk.

**TH.** ตารางเมตริกต่อคู่ (MAE / RMSE / Bias / Peak-hour MAE) + bar chart MAE ต่อสถานี — ต้องโชว์ inline ใน notebook (ตาม requirement: ทุกครั้งที่เทรนต้องเห็น metrics + charts ใน notebook ไม่ใช่บนดิสก์อย่างเดียว).

In [ ]:
# --- assemble metrics table from manifest (so resumed runs include prior pairs) ---
import json
import pandas as pd

manifest_now = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
rows = []
for (sid, h) in RUNS:
    entry = manifest_now["pairs"].get(f"{sid}_h{h}", {})
    metrics = entry.get("metrics", {}) or {}
    rows.append({
        "station_id": sid,
        "horizon_h": h,
        "status": entry.get("status", "pending"),
        "MAE":     metrics.get("mae"),
        "RMSE":    metrics.get("rmse"),
        "Bias":    metrics.get("bias"),
        "Peak_MAE": metrics.get("peak_mae"),
        "n_train": metrics.get("n_train"),
        "n_test":  metrics.get("n_test"),
        "train_s": metrics.get("train_seconds"),
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RUN_DIR / "metrics_table.csv", index=False)
metrics_df

In [ ]:
# --- per-station MAE bar chart (one bar per (station, horizon)) ---------
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plot_df = metrics_df.dropna(subset=["MAE"]).copy()
if plot_df.empty:
    print("No MAE values yet — train at least one pair to see the chart.")
else:
    stations = sorted(plot_df["station_id"].unique())
    horizons = sorted(plot_df["horizon_h"].unique())
    width = 0.8 / max(len(horizons), 1)
    fig, ax = plt.subplots(figsize=(max(6, len(stations) * 1.2 + 2), 4.5))
    x_idx = np.arange(len(stations))
    for k, h in enumerate(horizons):
        sub = plot_df[plot_df["horizon_h"] == h].set_index("station_id")
        vals = [sub.loc[s, "MAE"] if s in sub.index else float("nan") for s in stations]
        ax.bar(x_idx + (k - (len(horizons) - 1) / 2) * width, vals, width=width, label=f"h{h}")
    ax.set_xticks(x_idx)
    ax.set_xticklabels(stations)
    ax.set_ylabel("MAE (°C)")
    ax.set_title(f"LightGBM baseline — MAE per station × horizon\nrun_id={RUN_ID}")
    ax.legend(title="horizon", fontsize=8, ncol=min(len(horizons), 5))
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    chart_path = RUN_DIR / "mae_by_station.png"
    plt.savefig(chart_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"saved chart: {chart_path}")

## What's next / ขั้นถัดไป

**EN.** With baseline LightGBM artifacts in place, continue with `04_train_quantile.ipynb` to fit the calibrated quantile heads (`q05/q50/q95/q97`) and update `choice_matrix.json` to `lightgbm_quantile` / `lightgbm_hi_quantile` for the slots where the quantile run beats this baseline. Push artifacts back to the local repo via the workflow in `docs/colab-training.md`.

**TH.** เมื่อ baseline LightGBM artifacts พร้อมแล้ว ไปต่อที่ `04_train_quantile.ipynb` เพื่อเทรน quantile heads (`q05/q50/q95/q97`) แบบ calibrated และอัปเดต `choice_matrix.json` เป็น `lightgbm_quantile` / `lightgbm_hi_quantile` เฉพาะคู่ที่ quantile ชนะ baseline. push artifacts กลับ local repo ตาม `docs/colab-training.md`.

**Resume note / หมายเหตุการ resume.** หาก Colab disconnect ระหว่างรันให้เปิด notebook ใหม่และรันทุก cell ตั้งแต่ต้น — pair ที่เสร็จแล้วจะถูกข้ามอัตโนมัติเพราะมีทั้ง `bundle.json` และ entry ใน `manifest.json`.